In [0]:
# Databricks notebook source
# =============================================================================
# SILVER LAYER : f_SalesOrder Transformation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

# ──────────────────────────────────────────────────────────────────────────────
# 1. ENVIRONMENT CONFIGURATION & OPTIMIZATIONS
# ──────────────────────────────────────────────────────────────────────────────
# Kill the small file problem automatically!
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_f_salesorder")
print("⚪ INITIALIZING PROD SILVER PIPELINE : SALES ORDER HEADER")

CATALOG_NAME = "prx"
BRONZE_SCHEMA = "prx_bronze"
SILVER_SCHEMA = "prx_silver"

# Fully Qualified Unity Catalog Table Names
BRONZE_TABLE     = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_salesorderheader"
DIM_BILL_TO      = f"{CATALOG_NAME}.{SILVER_SCHEMA}.custbilltoaddr"
DIM_SHIP_TO      = f"{CATALOG_NAME}.{SILVER_SCHEMA}.custshiptoaddr"
DIM_CUSTOMER     = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customer"

SILVER_TABLE     = f"{CATALOG_NAME}.{SILVER_SCHEMA}.salesorder"
EXCEPTION_TABLE  = f"{CATALOG_NAME}.{SILVER_SCHEMA}.central_exceptions"

# External Paths for initial creation
STORAGE_ACCOUNT  = "stpraxaslakehouse"
SILVER_PATH      = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/f_salesorder"
EXCEPTION_PATH   = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/central_exceptions"

MERGE_KEYS       = ["SOSrcID"]

# Ensure Silver Schema Exists Securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Capture true executing user for Auditing
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# ──────────────────────────────────────────────────────────────────────────────
# 2. UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def safe_str(c): 
    return F.coalesce(F.col(c).cast(StringType()), F.lit(""))

def safe_float(c): 
    return F.coalesce(F.col(c).cast(FloatType()), F.lit(0.0))

def empty_str_to_null_ts(c):
    return F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), None)\
            .otherwise(F.col(c)).cast(TimestampType())

def coalesce_chain(*cols):
    """Helper to replicate COALESCE / CASE WHEN logic for falling back between joined columns"""
    expr = F.when(safe_str(cols[0]) != "", F.col(cols[0]))
    for c in cols[1:]:
        expr = expr.when(safe_str(c) != "", F.col(c))
    return expr.otherwise(F.lit(""))

# ──────────────────────────────────────────────────────────────────────────────
# 3. READ SOURCE & DIMENSIONS (BROADCAST JOINS)
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Reading Source & Dimension Data from Unity Catalog...")

# Base Source
bronze_df = spark.table(BRONZE_TABLE).alias("a")

# Dimensions (Filtered & Deduplicated)
dim_bill_to = spark.table(DIM_BILL_TO).filter(F.col("Main") == "true").dropDuplicates(["CustSrcID"]).alias("b")
dim_ship_to = spark.table(DIM_SHIP_TO).dropDuplicates(["CustShipToAddrSrcId"]).alias("c")
dim_customer = spark.table(DIM_CUSTOMER).dropDuplicates(["CustSrcID"]).alias("d")

# Extract the maximum FX Rate globally as a scalar for the Post Update fallback
latest_fx_rate = 1.0

logger.info("Applying Broadcast Joins to Dimensions...")
joined_df = bronze_df \
    .join(F.broadcast(dim_bill_to), (F.col("a.InvoiceTo") == F.col("b.CustSrcID")) , "left") \
    .join(F.broadcast(dim_ship_to), (F.col("a.DeliveryAddress") == F.col("c.CustShipToAddrSrcId")), "left") \
    .join(F.broadcast(dim_customer), (F.col("a.OrderedBy") == F.col("d.CustSrcID")), "left") 

# ──────────────────────────────────────────────────────────────────────────────
# 4. TRANSFORM & EXPAND COLUMNS
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Applying Complex Business Transformations...")

# Pre-calculate complex expressions
so_status = F.when(F.col("a.StatusDescription") == "Complete", "Completed") \
             .when(F.col("a.StatusDescription").isin("Open", "Partial"), "Open") \
             .when(F.col("a.StatusDescription") == "Cancelled", "Cancelled") \
             .otherwise("")

# Exchange Rate complex calculation
exrt_calc = F.when(F.col("a.Currency") == "Eur", F.lit(1.0)) \
             .when((safe_float("a.AmountDC") == 0) & (safe_float("a.AmountFC") == 0), F.lit(1.0)) \
             .when(safe_float("a.AmountDC") == 0, F.lit(1.0)) \
             .otherwise(F.coalesce(safe_float("a.AmountFC"), F.lit(1.0)) / F.coalesce(safe_float("a.AmountDC"), F.lit(1.0)))
exrt_calc_final = F.coalesce(exrt_calc, F.lit(1.0))

silver_df = joined_df.select(
    # Core Identifiers
    safe_str("a.OrderID").alias("SOSrcID"),
    safe_str("a.SalesPerson").alias("SalesRepInSrcID"),
    safe_str("a.SalesPerson").alias("SalesRepOutSrcID"),
    safe_str("a.OrderedBy").alias("CustSrcID"),
    safe_str("a.OrderedByContactPerson").alias("CustContactSrcID"),
    F.lit("").alias("CustBilltoAddrSrcID"),
    safe_str("a.DeliveryAddress").alias("CustShipToAddrSrcID"),
    
    # Order Keys
    safe_str("a.OrderNumber").alias("SOKey"),
    F.lit(0).alias("SOSeqKey"),
    F.lit("SO").alias("SOTrxnType"),
    F.lit("SalesOrder").alias("SrcSOTrxnType"),
    F.lit("SalesOrder").alias("SOTrxnTypeDesc"),
    F.lit("S").alias("SOType"),
    F.lit("Standard").alias("SOTypeDesc"),
    F.lit("Life Sciences").alias("SOOrdCat"),
    F.lit("Life Sciences").alias("SOOrdCatDesc"),
    F.lit("Life Sciences").alias("SOSrcOrdCat"),
    
    # Statuses
    safe_str("a.StatusDescription").alias("SOProcessStatus"),
    so_status.alias("SOStatus"),
    F.lit("").alias("SOShipStatus"),
    
    # Dates
    empty_str_to_null_ts("a.OrderDate").alias("SODt"),
    empty_str_to_null_ts("a.DeliveryDate").alias("ShipDt"),
    F.lit(None).cast(TimestampType()).alias("ReqShipDt"),
    F.when(so_status == "Completed", empty_str_to_null_ts("a.Modified")).otherwise(F.lit(None)).cast(TimestampType()).alias("CompletedDt"),
    F.when(so_status == "Cancelled", empty_str_to_null_ts("a.Modified")).otherwise(F.lit(None)).cast(TimestampType()).alias("CancDt"),

    # Representatives
    safe_str("a.SalesPerson").alias("SalesRepInKey"),
    safe_str("a.SalesPerson").alias("SalesRepOutKey"),
    safe_str("a.SalesPersonFullName").alias("SalesRepName"),
    F.lit("").alias("SalesRepTeam"),
    
    # Customer Details
    F.lit("").alias("CustDivKey"),
    F.coalesce(F.trim(F.col("d.CustKey")), F.lit("")).alias("CustKey"),
    F.lit("").alias("CustPOKey"),
    F.lit("").alias("CustFPKey"),
    safe_str("a.OrderedByName").alias("CustName"),
    safe_str("a.OrderedByContactPersonFullName").alias("CustContactName"),
    
    # Bill To (Cascading logic b -> d -> default)
    safe_str("b.CustBillToAddrKey").alias("CustBillToAddrKey"),
    coalesce_chain("b.CustName", "d.CustName").alias("CustBillToName"),
    coalesce_chain("b.AddrLn1").alias("CustBillToAddrLn1"),
    coalesce_chain("b.AddrLn2").alias("CustBillToAddrLn2"),
    coalesce_chain("b.City", "d.City").alias("CustBillToCity"),
    coalesce_chain("b.State", "d.State").alias("CustBillToState"),
    coalesce_chain("b.Country", "d.Country").alias("CustBillToCountry"),
    coalesce_chain("b.ZipCd", "d.ZipCd").alias("CustBillToZipCd"),
    
    # Ship To
    safe_str("a.DeliveryAddress").alias("CustShipToAddrKey"),
    safe_str("a.DeliverToContactPerson").alias("CustShipToContactKey"),
    coalesce_chain("a.DeliverToName", "d.CustName").alias("CustShipToName"),
    safe_str("a.DeliverToContactPersonFullName").alias("CustShipToContactName"),
    coalesce_chain("c.AddrLn1").alias("CustShipToAddrLn1"),
    coalesce_chain("c.AddrLn2").alias("CustShipToAddrLn2"),
    coalesce_chain("c.City", "d.City").alias("CustShipToCity"),
    coalesce_chain("c.State", "d.State").alias("CustShipToState"),
    coalesce_chain("c.Country", "d.Country").alias("CustShipToCountry"),
    coalesce_chain("c.ZipCd", "d.ZipCd").alias("CustShipToZipCd"),
    
    # Sell To
    safe_str("b.CustBillToAddrKey").alias("CustSellToAddrKey"),
    coalesce_chain("b.CustName", "d.CustName").alias("CustSellToName"),
    safe_str("b.CustContactSrcId").alias("CustSellToContactKey"),
    safe_str("b.CustContactName").alias("CustSellToContactName"),
    coalesce_chain("b.AddrLn1").alias("CustSellToAddrLn1"),
    coalesce_chain("b.AddrLn2").alias("CustSellToAddrLn2"),
    coalesce_chain("b.City", "d.City").alias("CustSellToCity"),
    coalesce_chain("b.State", "d.State").alias("CustSellToState"),
    coalesce_chain("b.Country", "d.Country").alias("CustSellToCountry"),
    coalesce_chain("b.ZipCd", "d.ZipCd").alias("CustSellToZipCd"),
    
    # Shipping & Payment
    safe_str("a.ShippingMethod").alias("ShipViaSrcID"),
    safe_str("a.ShippingMethod").alias("ShipViaKey"),
    safe_str("a.ShippingMethodDescription").alias("ShipVia"),
    F.lit("").alias("ShipInstr"),
    safe_str("a.PaymentCondition").alias("TermsCd"),
    safe_str("a.PaymentConditionDescription").alias("TermsDesc"),
    
    # Currency & FX 
    safe_str("a.Currency").alias("CustCurr"),
    F.lit("EUR").alias("LocCurr"),
    empty_str_to_null_ts("a.OrderDate").alias("ExDt"),
    exrt_calc_final.alias("ExRtLCCC"),
    (F.lit(1.0) / exrt_calc_final).alias("ExRtCCLC"),
    F.lit(latest_fx_rate).alias("ExRtLCUS"),
    
    # Financial Values & Tax
    F.lit(0.0).alias("SOTaxValCC"),
    F.lit(0.0).alias("SOTaxValLC"),
    F.lit(0.0).alias("SOTaxValUS"),
    
    # Additional Fields
    safe_str("a.Description").alias("AddInfo"),
    safe_str("a.Division").alias("CompanyDivKey"),
    safe_str("a.Remarks").alias("Comment"),
    safe_str("a.Document").alias("SODocID"),
    safe_str("a.DocumentNumber").alias("SODocKey"),
    safe_str("a.IncotermCode").alias("IncoTermCd"),
    safe_str("a.SalesChannel").alias("SalesChnlID"),
    F.lit("SalesOrderHeader").alias("SrcTable"),
    F.lit("No").alias("IsDeletedFl"),

    # Source Audit
    empty_str_to_null_ts("a.Created").alias("CreatedDt"),
    safe_str("a.CreatorFullName").alias("CreatedBy"),
    empty_str_to_null_ts("a.Modified").alias("UpdatedDt"),
    safe_str("a.ModifierFullName").alias("UpdatedBy"),
    
    # Enterprise Pipeline Audit
    F.current_timestamp().alias("SysCreatedDt"),
    F.current_timestamp().alias("SysUpdatedDt"),
    F.lit(executing_user).alias("SysUpdatedBy")
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. DATA QUALITY & DEDUPLICATION
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Evaluating Data Quality Rules...")

dq_df = silver_df.withColumn(
    "DQ_Reason",
    F.array_remove(F.array(
        F.when(F.col("SOSrcID") == "", F.lit("MISSING_SO_ID")),
        F.when(F.col("SOKey") == "", F.lit("MISSING_SO_KEY"))
    ), None)
).withColumn(
    "DQ_Status",
    F.when(F.size("DQ_Reason") > 0, F.lit(2)).otherwise(F.lit(0))
)

window_dup = Window.partitionBy("SOSrcID").orderBy(F.col("UpdatedDt").desc_nulls_last())

dq_df = dq_df.withColumn(
    "rn", F.row_number().over(window_dup)
).withColumn(
    "DQ_Status",
    F.when(F.col("rn") > 1, F.lit(4)).otherwise(F.col("DQ_Status"))
).drop("rn")

# Cache to prevent DAG explosion during counts & routing
dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status").isin(2, 4))

valid_count = valid_df.count()
error_count = error_df.count()

logger.info(f"✅ Valid Records: {valid_count:,}")
logger.info(f"❌ Error Records: {error_count:,}")

# ──────────────────────────────────────────────────────────────────────────────
# 6. CENTRAL EXCEPTION ROUTING
# ──────────────────────────────────────────────────────────────────────────────
if error_count > 0:
    logger.info("Routing exceptions to Unity Catalog Quarantine...")
    exception_df = error_df.select(
        F.lit(SILVER_TABLE).alias("table_name"),
        F.col("SOSrcID").alias("record_key"), 
        F.when(F.col("DQ_Status") == 4, F.lit("Duplicate Record"))
         .otherwise(F.concat(F.lit("Validation Failed: "), F.concat_ws(", ", F.col("DQ_Reason")))).alias("exception_details"),
        F.current_timestamp().alias("syscreateddt")
    )

    if spark.catalog.tableExists(EXCEPTION_TABLE):
        exception_df.write.format("delta").mode("append").saveAsTable(EXCEPTION_TABLE)
    else:
        exception_df.write.format("delta").mode("overwrite").option("path", EXCEPTION_PATH).saveAsTable(EXCEPTION_TABLE)

# ──────────────────────────────────────────────────────────────────────────────
# 7. MERGE INTO SILVER DELTA
# ──────────────────────────────────────────────────────────────────────────────
if valid_count > 0:
    logger.info("Executing MERGE INTO Silver Delta Table...")
    window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("UpdatedDt").desc_nulls_last())
    final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    if spark.catalog.tableExists(SILVER_TABLE):
        delta_tbl = DeltaTable.forName(spark, SILVER_TABLE)
        cond = " AND ".join([f"tgt.{k} = src.{k}" for k in MERGE_KEYS])
        
        (delta_tbl.alias("tgt")
         .merge(final_df.alias("src"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        final_df.write.format("delta").mode("overwrite").option("path", SILVER_PATH).saveAsTable(SILVER_TABLE)

dq_df.unpersist()
print("🎉 PROD SALES ORDER PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM prx.prx_silver.salesorder LIMIT 10;